# Chapter 4 — Paradigms of Machine Learning II: Semi-Supervised, Active, Self-Supervised, Online, Meta & Reinforcement Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part1_foundations/ch04_paradigms_of_machine_learning_ii/ch04_notebook.ipynb)

Companion notebook for **Chapter 4** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Semi-supervised learning adds unlabelled data to a few labels, often through confident pseudo-labels.
- Active learning spends the labelling budget on the examples the model is least sure about.
- Self-supervised learning creates its own labels from a pretext task, then reuses what it learned downstream.
- Online learning updates continuously and can adapt when the data drifts.
- Meta-learning trains on many tasks so that new ones need only a few examples (N-way K-shot, zero-shot).
- Reinforcement learning has no labels. An agent learns a policy from rewards by trial and error.

## Example 1: Semi-Supervised Learning with Pseudo-Labels

Only 50 of 1,257 digit images keep their labels. A model trained on those 50 scores 82.6%. Then the self-training loop runs. Each round, the model labels the unlabelled images it is at least 90% sure about and retrains on them. Within four rounds it has pseudo-labelled over 1,000 images and test accuracy rises to 88.1%. Label spreading, a second semi-supervised method, passes labels to neighbouring images and reaches 92.8% from the same 50 labels. Compare these with the 96.3% you get when all 1,257 labels are available.

In [1]:
# Chapter 4 - Example 1: Semi-supervised learning with pseudo-labels (self-training)
# Only 50 of ~1,250 training images have labels. The model labels the rest itself,
# keeps only the labels it is very confident about, and retrains.
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading

X, y = load_digits(return_X_y=True)
X = X / 16.0
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

rng = np.random.default_rng(0)
labelled = np.zeros(len(X_tr), bool)
for d in range(10):                                  # 5 labelled images per digit
    labelled[rng.choice(np.where(y_tr == d)[0], 5, replace=False)] = True
X_l, y_l = X_tr[labelled], y_tr[labelled]
X_u = X_tr[~labelled]                                # unlabelled pool (we ignore y for these)
print(f"Labelled: {len(X_l)}   Unlabelled: {len(X_u)}")

def new_model():
    return LogisticRegression(C=100, max_iter=3000)

model = new_model().fit(X_l, y_l)
print(f"Round 0 (labelled data only): test accuracy = {model.score(X_te, y_te):.3f}")

for rnd in range(1, 5):
    proba = model.predict_proba(X_u)
    confident = proba.max(axis=1) >= 0.9             # keep only very confident guesses
    pseudo = model.classes_[proba.argmax(axis=1)]
    model = new_model().fit(np.vstack([X_l, X_u[confident]]),
                            np.r_[y_l, pseudo[confident]])
    true_u = y_tr[~labelled][confident]               # peek ONLY to report pseudo-label quality
    print(f"Round {rnd}: pseudo-labelled {confident.sum():4d} images "
          f"(pseudo-label accuracy {np.mean(pseudo[confident] == true_u):.3f}) "
          f"-> test accuracy = {model.score(X_te, y_te):.3f}")

# A second semi-supervised method: label spreading passes labels along to near neighbours
y_partial = np.where(labelled, y_tr, -1)             # -1 means 'no label'
ls = LabelSpreading(kernel="knn", n_neighbors=10).fit(X_tr, y_partial)
print(f"\nLabel spreading (same 50 labels): test accuracy = {ls.score(X_te, y_te):.3f}")

full = new_model().fit(X_tr, y_tr)
print(f"For reference, with ALL {len(X_tr)} true labels: {full.score(X_te, y_te):.3f}")

Labelled: 50   Unlabelled: 1207
Round 0 (labelled data only): test accuracy = 0.826
Round 1: pseudo-labelled  727 images (pseudo-label accuracy 0.961) -> test accuracy = 0.859
Round 2: pseudo-labelled 1010 images (pseudo-label accuracy 0.914) -> test accuracy = 0.872
Round 3: pseudo-labelled 1079 images (pseudo-label accuracy 0.904) -> test accuracy = 0.878
Round 4: pseudo-labelled 1091 images (pseudo-label accuracy 0.902) -> test accuracy = 0.881

Label spreading (same 50 labels): test accuracy = 0.928
For reference, with ALL 1257 true labels: 0.963


## Example 2: Active Learning vs Random Labelling

Both strategies start with 20 labelled images and add 10 labels per round. The random strategy labels any 10 images. The active strategy asks the 'expert' to label the 10 images where the model's top two guesses are closest, which is uncertainty sampling. Averaged over five runs, active learning pulls ahead from the first round. With 100 labels it scores 92.7% against 88.6%, so the same labelling budget buys a better model.

In [2]:
# Chapter 4 - Example 2: Active learning - ask the expert to label the MOST useful images
# Uncertainty sampling picks the images the model is least sure about.
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

X, y = load_digits(return_X_y=True)
X = X / 16.0
X_pool, X_te, y_pool, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=1)

def run(strategy, seed=0, start=20, batch=10, rounds=8):
    rng = np.random.default_rng(seed)
    labelled = list(rng.choice(len(X_pool), start, replace=False))
    scores = []
    for _ in range(rounds + 1):
        m = LogisticRegression(max_iter=2000).fit(X_pool[labelled], y_pool[labelled])
        scores.append(m.score(X_te, y_te))
        rest = np.setdiff1d(np.arange(len(X_pool)), labelled)
        if strategy == "uncertainty":
            p = np.sort(m.predict_proba(X_pool[rest]), axis=1)
            margin = p[:, -1] - p[:, -2]              # small gap = model is unsure
            pick = rest[np.argsort(margin)[:batch]]
        else:
            pick = rng.choice(rest, batch, replace=False)
        labelled += list(pick)                        # the 'expert' supplies these labels
    return np.array(scores)

active = np.mean([run("uncertainty", s) for s in range(5)], axis=0)
random_ = np.mean([run("random", s) for s in range(5)], axis=0)
print("Labels used   Random sampling   Active (uncertainty)")
for i, (r, a) in enumerate(zip(random_, active)):
    print(f"   {20 + 10 * i:3d}           {r:.3f}             {a:.3f}")
print(f"\nWith 100 labels, active learning is {100 * (active[-1] - random_[-1]):.1f} "
      "percentage points more accurate (average of 5 runs).")

Labels used   Random sampling   Active (uncertainty)
    20           0.550             0.550
    30           0.707             0.731
    40           0.758             0.824
    50           0.782             0.877
    60           0.820             0.883
    70           0.845             0.901
    80           0.857             0.921
    90           0.870             0.925
   100           0.886             0.927

With 100 labels, active learning is 4.1 percentage points more accurate (average of 5 runs).


## Example 3: Online Learning When the World Changes

This simulates a data stream where the rule suddenly flips at batch 10, which is concept drift. The static model was trained once at the start and never updated, so after the drift it falls to coin-flip accuracy (about 50%). The online model is updated with partial_fit after every batch. It stumbles for one batch, then recovers to about 97%. Note the order inside the loop: each batch is first used to test, then to learn, exactly as in a live system.

In [3]:
# Chapter 4 - Example 3: Online learning on a data stream that changes (concept drift)
# A model trained once goes stale; a model updated batch-by-batch keeps up.
import numpy as np
from sklearn.linear_model import SGDClassifier

rng = np.random.default_rng(0)

def make_batch(t, n=200):
    """Two features. Before t=10 the rule is x1 + x2 > 0; after t=10 the rule flips."""
    X = rng.normal(0, 1, (n, 2))
    y = (X[:, 0] + X[:, 1] > 0).astype(int)
    if t >= 10:
        y = (X[:, 0] - X[:, 1] > 0).astype(int)   # the world has changed (drift)
    return X, y

static = SGDClassifier(loss="log_loss", random_state=0)
online = SGDClassifier(loss="log_loss", random_state=0)
X0, y0 = make_batch(0)
static.fit(X0, y0)                                  # trained once, then frozen
online.partial_fit(X0, y0, classes=[0, 1])

print("Batch   Static model   Online model")
for t in range(1, 20):
    X, y = make_batch(t)
    s_acc, o_acc = static.score(X, y), online.score(X, y)  # test on new data first...
    online.partial_fit(X, y)                               # ...then learn from it
    if t % 2 == 1 or t in (10, 11):
        flag = "  <- drift happens here" if t == 10 else ""
        print(f"  {t:2d}      {s_acc:.2f}          {o_acc:.2f}{flag}")

Batch   Static model   Online model
   1      0.99          0.95
   3      0.98          0.98
   5      0.96          0.97
   7      0.98          0.98
   9      0.98          0.98
  10      0.53          0.52  <- drift happens here
  11      0.50          0.97
  13      0.48          0.98
  15      0.49          0.98
  17      0.51          0.96
  19      0.57          0.98
